# Prevalence Stress Test

This notebook is Milestone 07 for the revised fraud-detection paper.

Research purpose:

- Train strict leakage-safe fraud models once on natural training data.
- Choose one cost-sensitive threshold on the natural validation set.
- Re-evaluate the fixed model and fixed threshold under shifted fraud prevalence in the test stream.
- Quantify how precision, MCC, AP, false alerts, missed fraud, and operating cost change as the fraud rate changes.

This is an operational robustness test, not a new model-comparison experiment.

In [ ]:
!pip -q install -U imbalanced-learn lightgbm xgboost seaborn
!pip -q install -q "pyarrow>=19,<24"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Drive screenshot showed: My Drive > Credit__ > Datasets
PROJECT_ROOT = Path('/content/drive/MyDrive/Credit__')

DATASET_SEARCH_DIRS = [
    PROJECT_ROOT / 'Datasets',
    PROJECT_ROOT / '00_source_datasets',
    PROJECT_ROOT,
]

RUN_NAME = 'prevalence_stress_v1'
EXPERIMENT_ROOT = PROJECT_ROOT / '07_prevalence_stress_test' / RUN_NAME
CHECKPOINT_DIR = EXPERIMENT_ROOT / 'checkpoints'
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
FIGURES_DIR = EXPERIMENT_ROOT / 'figures'
LOGS_DIR = EXPERIMENT_ROOT / 'logs'

for d in [EXPERIMENT_ROOT, CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 101, 202, 303, 404]
MODEL_NAMES = ['xgboost', 'lightgbm']

TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20

TARGET_PREVALENCES = [0.0005, 0.0010, 0.0020, 0.0050, 0.0100]
PREVALENCE_REPEATS = 5
MAX_EVAL_ROWS_PER_PREVALENCE = 100_000
MIN_FRAUDS_PER_EVAL = 20

CAPS = {
    'D1_creditcard': None,
    'D2_fraudTest': None,
    'D3_PaySim': 1_000_000,
}

SMOKE_TEST = False
SMOKE_CAP = 50_000

print('Project root:', PROJECT_ROOT)
print('Experiment root:', EXPERIMENT_ROOT)
print('Seeds:', SEEDS)

In [ ]:
import json
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except Exception as exc:
    xgb = None
    print('XGBoost unavailable:', exc)

try:
    import lightgbm as lgb
except Exception as exc:
    lgb = None
    print('LightGBM unavailable:', exc)

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'legend.title_fontsize': 8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

def now():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def write_csv_atomic(path, df):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def append_log(message):
    line = f'[{now()}] {message}'
    print(line)
    with open(LOGS_DIR / 'run.log', 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def checkpoint_path(seed, dataset, model_name):
    return CHECKPOINT_DIR / f'seed{seed}__{dataset}__{model_name}.csv'

def is_done(seed, dataset, model_name):
    return checkpoint_path(seed, dataset, model_name).exists()

In [ ]:
def find_dataset_file(file_name):
    candidates = []
    for directory in DATASET_SEARCH_DIRS:
        candidates.append(directory / file_name)
        if directory.exists():
            candidates.extend(directory.glob(f'**/{file_name}'))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not find {file_name}')

DATASET_FILES = {
    'D1_creditcard': find_dataset_file('creditcard.csv'),
    'D2_fraudTest': find_dataset_file('fraudTest.csv'),
    'D3_PaySim': find_dataset_file('PS.csv'),
}

for name, path in DATASET_FILES.items():
    append_log(f'{name}: {path}')

write_json(EXPERIMENT_ROOT / 'dataset_file_manifest.json', {k: str(v) for k, v in DATASET_FILES.items()})

In [ ]:
def temporal_subsample(df, order_col, cap):
    if SMOKE_TEST:
        cap = min(SMOKE_CAP, len(df))
    if cap is None or len(df) <= cap:
        return df.reset_index(drop=True)
    positions = np.linspace(0, len(df) - 1, num=cap, dtype=np.int64)
    return df.iloc[positions].reset_index(drop=True)

def load_dataset(dataset_name):
    path = DATASET_FILES[dataset_name]
    df = pd.read_csv(path, low_memory=False)

    if dataset_name == 'D1_creditcard':
        df = df.drop_duplicates().reset_index(drop=True)
        label_col = 'Class'
        df['__order__'] = pd.to_numeric(df['Time'], errors='coerce')

    elif dataset_name == 'D2_fraudTest':
        label_col = 'is_fraud'
        df['__order__'] = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')

    elif dataset_name == 'D3_PaySim':
        label_col = 'isFraud'
        df = df.dropna(subset=[label_col]).reset_index(drop=True)
        df[label_col] = df[label_col].astype(int)
        df['__order__'] = pd.to_numeric(df['step'], errors='coerce')

    else:
        raise ValueError(dataset_name)

    df = df.dropna(subset=[label_col]).sort_values('__order__').reset_index(drop=True)
    df = temporal_subsample(df, '__order__', CAPS[dataset_name])
    fraud = int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum())
    append_log(f'{dataset_name}: rows={len(df):,}, fraud={fraud:,}, rate={fraud/len(df)*100:.4f}%')
    return df, label_col

def prepare_features(df, dataset_name):
    df = df.copy()

    if dataset_name == 'D1_creditcard':
        y = df['Class'].astype(int)
        X = df[[f'V{i}' for i in range(1, 29)] + ['Amount']].copy()
        return X, y

    if dataset_name == 'D2_fraudTest':
        df['trans_dt'] = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')
        df['dob_dt'] = pd.to_datetime(df['dob'], errors='coerce')
        df['trans_hour'] = df['trans_dt'].dt.hour
        df['trans_dayofweek'] = df['trans_dt'].dt.dayofweek
        df['trans_month'] = df['trans_dt'].dt.month
        df['age_years'] = (df['trans_dt'] - df['dob_dt']).dt.days / 365.25
        keep = ['category', 'amt', 'state', 'city_pop', 'trans_hour', 'trans_dayofweek', 'trans_month', 'age_years']
        return df[keep].copy(), df['is_fraud'].astype(int)

    if dataset_name == 'D3_PaySim':
        keep = ['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']
        return df[keep].copy(), df['isFraud'].astype(int)

    raise ValueError(dataset_name)

def make_preprocessor(X):
    numeric_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler(with_centering=False)),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=True)),
    ])
    return ColumnTransformer([
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols),
    ])

In [ ]:
def build_model(model_name, seed, scale_pos_weight=1.0):
    if model_name == 'xgboost':
        if xgb is None:
            raise RuntimeError('XGBoost not available')
        return xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            objective='binary:logistic',
            eval_metric='aucpr',
            tree_method='hist',
            device='cuda',
            scale_pos_weight=scale_pos_weight,
            random_state=seed,
            n_jobs=-1,
        )
    if model_name == 'lightgbm':
        if lgb is None:
            raise RuntimeError('LightGBM not available')
        return lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.04,
            num_leaves=63,
            class_weight='balanced',
            random_state=seed,
            n_jobs=-1,
            verbose=-1,
        )
    raise ValueError(model_name)

def get_scores(model, X):
    return model.predict_proba(X)[:, 1]

def choose_threshold(y_val, scores_val, c_fn=100, c_fp=1):
    best = None
    for tau in np.round(np.arange(0.01, 0.91, 0.01), 2):
        pred = (scores_val >= tau).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
        cost = c_fp * fp + c_fn * fn
        mcc = matthews_corrcoef(y_val, pred) if len(np.unique(pred)) > 1 else 0.0
        candidate = (cost, fn, fp, -mcc, tau)
        if best is None or candidate < best:
            best = candidate
    return float(best[-1])

def evaluate_predictions(y_true, scores, tau):
    pred = (scores >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    n = len(y_true)
    return {
        'threshold': float(tau),
        'n_eval': int(n),
        'eval_fraud': int(np.sum(y_true == 1)),
        'eval_legit': int(np.sum(y_true == 0)),
        'eval_prevalence': float(np.mean(y_true == 1)),
        'tp': int(tp),
        'fp': int(fp),
        'tn': int(tn),
        'fn': int(fn),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, pred)) if len(np.unique(pred)) > 1 else 0.0,
        'average_precision': float(average_precision_score(y_true, scores)),
        'roc_auc': float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) > 1 else np.nan,
        'brier': float(brier_score_loss(y_true, scores)),
        'false_alerts_per_10k': float(fp / n * 10_000),
        'missed_frauds_per_10k': float(fn / n * 10_000),
        'cost_per_10k': float((fp + 100 * fn) / n * 10_000),
    }

In [ ]:
def split_train_val_test(X, y, seed):
    idx = np.arange(len(y))
    train_val_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, stratify=y, random_state=seed)
    y_train_val = y.iloc[train_val_idx]
    val_relative = VALIDATION_SIZE / (1.0 - TEST_SIZE)
    train_idx, val_idx = train_test_split(train_val_idx, test_size=val_relative, stratify=y_train_val, random_state=seed)
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)

def train_and_score_test_pool(X, y, model_name, seed):
    train_idx, val_idx, test_idx = split_train_val_test(X, y, seed)
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

    preprocessor = make_preprocessor(X_train)
    X_train_t = preprocessor.fit_transform(X_train)
    X_val_t = preprocessor.transform(X_val)
    X_test_t = preprocessor.transform(X_test)

    pos = max(int(np.sum(y_train == 1)), 1)
    neg = max(int(np.sum(y_train == 0)), 1)
    model = build_model(model_name, seed, scale_pos_weight=neg / pos)
    t0 = time.time()
    model.fit(X_train_t, y_train)
    fit_seconds = time.time() - t0

    val_scores = get_scores(model, X_val_t)
    threshold = choose_threshold(y_val, val_scores)
    test_scores = get_scores(model, X_test_t)

    return {
        'threshold': threshold,
        'fit_seconds': float(fit_seconds),
        'y_test': y_test.reset_index(drop=True),
        'test_scores': np.asarray(test_scores),
        'train_n': int(len(y_train)),
        'train_fraud': int(np.sum(y_train == 1)),
        'val_n': int(len(y_val)),
        'val_fraud': int(np.sum(y_val == 1)),
        'test_n': int(len(y_test)),
        'test_fraud': int(np.sum(y_test == 1)),
    }

def sample_prevalence_indices(y, target_prevalence, seed, repeat):
    y = pd.Series(y).reset_index(drop=True)
    pos_idx = np.flatnonzero(y.values == 1)
    neg_idx = np.flatnonzero(y.values == 0)
    rng = np.random.default_rng(seed + repeat * 10_000 + int(target_prevalence * 1_000_000))

    max_n_by_pos = int(len(pos_idx) / target_prevalence)
    max_n_by_neg = int(len(neg_idx) / (1.0 - target_prevalence))
    eval_n = min(MAX_EVAL_ROWS_PER_PREVALENCE, max_n_by_pos, max_n_by_neg, len(y))
    eval_n = max(eval_n, 0)
    n_pos = int(round(eval_n * target_prevalence))
    n_pos = max(n_pos, MIN_FRAUDS_PER_EVAL)
    n_pos = min(n_pos, len(pos_idx))
    n_neg = int(round(n_pos * (1.0 - target_prevalence) / target_prevalence))
    n_neg = min(n_neg, len(neg_idx), MAX_EVAL_ROWS_PER_PREVALENCE - n_pos)

    if n_pos < MIN_FRAUDS_PER_EVAL or n_neg <= 0:
        return None

    chosen_pos = rng.choice(pos_idx, size=n_pos, replace=False)
    chosen_neg = rng.choice(neg_idx, size=n_neg, replace=False)
    chosen = np.concatenate([chosen_pos, chosen_neg])
    rng.shuffle(chosen)
    return chosen

def evaluate_prevalence_grid(y_test, test_scores, threshold, seed):
    rows = []
    natural_metrics = evaluate_predictions(y_test, test_scores, threshold)
    rows.append({
        'prevalence_scenario': 'NATURAL_TEST_POOL',
        'target_prevalence': float(np.mean(y_test == 1)),
        'repeat': 0,
        'sample_status': 'natural',
        **natural_metrics,
    })

    for target in TARGET_PREVALENCES:
        for repeat in range(PREVALENCE_REPEATS):
            idx = sample_prevalence_indices(y_test, target, seed, repeat)
            if idx is None:
                rows.append({
                    'prevalence_scenario': f'target_{target:.4f}',
                    'target_prevalence': target,
                    'repeat': repeat,
                    'sample_status': 'skipped_insufficient_fraud_or_legit',
                })
                continue
            metrics = evaluate_predictions(y_test.iloc[idx].reset_index(drop=True), test_scores[idx], threshold)
            rows.append({
                'prevalence_scenario': f'target_{target:.4f}',
                'target_prevalence': target,
                'repeat': repeat,
                'sample_status': 'sampled',
                **metrics,
            })
    return rows

In [ ]:
def run_one(seed, dataset_name, model_name):
    if is_done(seed, dataset_name, model_name):
        append_log(f'SKIP seed={seed} {dataset_name} / {model_name}')
        return pd.read_csv(checkpoint_path(seed, dataset_name, model_name))

    df, label_col = load_dataset(dataset_name)
    X, y = prepare_features(df, dataset_name)
    trained = train_and_score_test_pool(X, y, model_name, seed)
    prevalence_rows = evaluate_prevalence_grid(
        trained['y_test'],
        trained['test_scores'],
        trained['threshold'],
        seed,
    )

    rows = []
    for row in prevalence_rows:
        out = {
            'run_name': RUN_NAME,
            'timestamp': now(),
            'seed': seed,
            'dataset': dataset_name,
            'model': model_name,
            'feature_policy': 'strict',
            'threshold_policy': 'validation_cost_minimization',
            'smoke_test': SMOKE_TEST,
            'rows_used': int(len(df)),
            'fraud_rows_used': int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum()),
            'cap_used': CAPS[dataset_name],
            'fit_seconds': trained['fit_seconds'],
            'train_n': trained['train_n'],
            'train_fraud': trained['train_fraud'],
            'val_n': trained['val_n'],
            'val_fraud': trained['val_fraud'],
            'test_pool_n': trained['test_n'],
            'test_pool_fraud': trained['test_fraud'],
            **row,
        }
        rows.append(out)

    result = pd.DataFrame(rows)
    write_csv_atomic(checkpoint_path(seed, dataset_name, model_name), result)
    natural = result[result['prevalence_scenario'] == 'NATURAL_TEST_POOL'].iloc[0]
    append_log(
        f'DONE seed={seed} {dataset_name} / {model_name}: '
        f'natural AP={natural["average_precision"]:.4f}, precision={natural["precision"]:.4f}, cost={natural["cost_per_10k"]:.2f}'
    )
    return result

all_frames = []
for seed in SEEDS:
    for dataset_name in ['D1_creditcard', 'D2_fraudTest', 'D3_PaySim']:
        for model_name in MODEL_NAMES:
            try:
                frame = run_one(seed, dataset_name, model_name)
                all_frames.append(frame)
                write_csv_atomic(RESULTS_DIR / 'partial_results_latest.csv', pd.concat(all_frames, ignore_index=True))
            except Exception as exc:
                append_log(f'ERROR seed={seed} {dataset_name} / {model_name}: {repr(exc)}')
                write_json(CHECKPOINT_DIR / f'ERROR__seed{seed}__{dataset_name}__{model_name}.json', {
                    'seed': seed,
                    'dataset': dataset_name,
                    'model': model_name,
                    'error': repr(exc),
                    'timestamp': now(),
                })

frames = [pd.read_csv(p) for p in sorted(CHECKPOINT_DIR.glob('seed*.csv'))]
results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
write_csv_atomic(RESULTS_DIR / 'prevalence_stress_results.csv', results)
results

In [ ]:
results = pd.read_csv(RESULTS_DIR / 'prevalence_stress_results.csv')
valid = results[results['sample_status'].isin(['natural', 'sampled'])].copy()

natural = valid[valid['prevalence_scenario'] == 'NATURAL_TEST_POOL'][[
    'seed', 'dataset', 'model',
    'average_precision', 'precision', 'mcc', 'recall', 'cost_per_10k',
    'false_alerts_per_10k', 'missed_frauds_per_10k', 'brier'
]].rename(columns={
    'average_precision': 'natural_average_precision',
    'precision': 'natural_precision',
    'mcc': 'natural_mcc',
    'recall': 'natural_recall',
    'cost_per_10k': 'natural_cost_per_10k',
    'false_alerts_per_10k': 'natural_false_alerts_per_10k',
    'missed_frauds_per_10k': 'natural_missed_frauds_per_10k',
    'brier': 'natural_brier',
})

index = valid.merge(natural, on=['seed', 'dataset', 'model'], how='left')
for metric in [
    'average_precision', 'precision', 'mcc', 'recall', 'cost_per_10k',
    'false_alerts_per_10k', 'missed_frauds_per_10k', 'brier'
]:
    index[f'delta_{metric}'] = index[metric] - index[f'natural_{metric}']

write_csv_atomic(RESULTS_DIR / 'prevalence_sensitivity_index.csv', index)

summary_by_prevalence = (
    index[index['sample_status'] == 'sampled']
    .groupby(['dataset', 'model', 'target_prevalence'])
    .agg(
        rows=('average_precision', 'size'),
        ap_mean=('average_precision', 'mean'),
        ap_std=('average_precision', 'std'),
        precision_mean=('precision', 'mean'),
        precision_std=('precision', 'std'),
        mcc_mean=('mcc', 'mean'),
        mcc_std=('mcc', 'std'),
        recall_mean=('recall', 'mean'),
        recall_std=('recall', 'std'),
        cost_mean=('cost_per_10k', 'mean'),
        cost_std=('cost_per_10k', 'std'),
        false_alerts_mean=('false_alerts_per_10k', 'mean'),
        missed_frauds_mean=('missed_frauds_per_10k', 'mean'),
    )
    .reset_index()
)
write_csv_atomic(RESULTS_DIR / 'prevalence_stress_summary_by_prevalence.csv', summary_by_prevalence)

sensitivity = (
    summary_by_prevalence
    .groupby(['dataset', 'model'])
    .agg(
        prevalence_levels=('target_prevalence', 'nunique'),
        ap_range=('ap_mean', lambda s: float(s.max() - s.min())),
        precision_range=('precision_mean', lambda s: float(s.max() - s.min())),
        mcc_range=('mcc_mean', lambda s: float(s.max() - s.min())),
        recall_range=('recall_mean', lambda s: float(s.max() - s.min())),
        cost_range=('cost_mean', lambda s: float(s.max() - s.min())),
        false_alerts_range=('false_alerts_mean', lambda s: float(s.max() - s.min())),
        missed_frauds_range=('missed_frauds_mean', lambda s: float(s.max() - s.min())),
    )
    .reset_index()
)
write_csv_atomic(RESULTS_DIR / 'prevalence_sensitivity_summary.csv', sensitivity)
sensitivity

In [ ]:
summary = pd.read_csv(RESULTS_DIR / 'prevalence_stress_summary_by_prevalence.csv')
sensitivity = pd.read_csv(RESULTS_DIR / 'prevalence_sensitivity_summary.csv')

dataset_labels = {'D1_creditcard': 'D1', 'D2_fraudTest': 'D2', 'D3_PaySim': 'D3'}
model_labels = {'xgboost': 'XGBoost', 'lightgbm': 'LightGBM'}
summary['dataset_label'] = summary['dataset'].map(dataset_labels)
summary['model_label'] = summary['model'].map(model_labels)
summary['prevalence_percent'] = summary['target_prevalence'] * 100
sensitivity['row_label'] = sensitivity['dataset'].map(dataset_labels) + ' - ' + sensitivity['model'].map(model_labels)

for metric, ylabel, filename in [
    ('precision_mean', 'Mean precision', 'prevalence_precision_curve'),
    ('mcc_mean', 'Mean MCC', 'prevalence_mcc_curve'),
    ('cost_mean', 'Mean cost per 10,000', 'prevalence_cost_curve'),
]:
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    sns.lineplot(
        data=summary,
        x='prevalence_percent',
        y=metric,
        hue='dataset_label',
        style='model_label',
        markers=True,
        dashes=False,
        linewidth=1.8,
        ax=ax,
    )
    ax.set_xlabel('Target fraud prevalence (%)')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + ' under prevalence shift')
    ax.grid(axis='both', linestyle='--', linewidth=0.45, alpha=0.55)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.legend(title='Dataset / model', loc='best', frameon=True)
    fig.tight_layout()
    png = FIGURES_DIR / f'{filename}.png'
    pdf = FIGURES_DIR / f'{filename}.pdf'
    fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    plt.show()
    append_log(f'Saved figure: {png}')

for metric, title, filename, cbar_label in [
    ('precision_range', 'Prevalence sensitivity in precision', 'prevalence_precision_sensitivity_heatmap', 'Precision range'),
    ('cost_range', 'Prevalence sensitivity in cost', 'prevalence_cost_sensitivity_heatmap', 'Cost range per 10,000'),
]:
    heat = sensitivity.pivot_table(index='row_label', values=metric, aggfunc='mean')
    fig, ax = plt.subplots(figsize=(4.8, max(3.0, 0.45 * len(heat))))
    sns.heatmap(
        heat,
        annot=True,
        fmt='.3f' if metric != 'cost_range' else '.1f',
        cmap='viridis',
        linewidths=0.5,
        linecolor='white',
        cbar_kws={'label': cbar_label},
        annot_kws={'fontsize': 8},
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Dataset and model')
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)
    fig.tight_layout()
    png = FIGURES_DIR / f'{filename}.png'
    pdf = FIGURES_DIR / f'{filename}.pdf'
    fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    plt.show()
    append_log(f'Saved figure: {png}')

In [ ]:
results = pd.read_csv(RESULTS_DIR / 'prevalence_stress_results.csv')
index = pd.read_csv(RESULTS_DIR / 'prevalence_sensitivity_index.csv')
sensitivity = pd.read_csv(RESULTS_DIR / 'prevalence_sensitivity_summary.csv')

run_summary = {
    'run_name': RUN_NAME,
    'completed_at': now(),
    'project_root': str(PROJECT_ROOT),
    'experiment_root': str(EXPERIMENT_ROOT),
    'seeds': SEEDS,
    'models': MODEL_NAMES,
    'target_prevalences': TARGET_PREVALENCES,
    'prevalence_repeats': PREVALENCE_REPEATS,
    'max_eval_rows_per_prevalence': MAX_EVAL_ROWS_PER_PREVALENCE,
    'min_frauds_per_eval': MIN_FRAUDS_PER_EVAL,
    'caps': CAPS,
    'results_file': str(RESULTS_DIR / 'prevalence_stress_results.csv'),
    'index_file': str(RESULTS_DIR / 'prevalence_sensitivity_index.csv'),
    'summary_by_prevalence_file': str(RESULTS_DIR / 'prevalence_stress_summary_by_prevalence.csv'),
    'sensitivity_summary_file': str(RESULTS_DIR / 'prevalence_sensitivity_summary.csv'),
    'figures_dir': str(FIGURES_DIR),
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'n_result_rows': int(len(results)),
    'n_index_rows': int(len(index)),
    'n_sensitivity_rows': int(len(sensitivity)),
}
write_json(EXPERIMENT_ROOT / 'run_summary.json', run_summary)
run_summary

## Expected Runtime on Colab T4

Expected runtime is approximately **35-70 minutes**.

Why:

- Only 30 model fits are required: 3 datasets x 2 models x 5 seeds.
- Prevalence stress is evaluation-only after each model is trained.
- D3 PaySim is capped to 1,000,000 timeline-sampled rows by default.

If Colab disconnects, rerun from the top. Existing checkpoint CSVs are skipped.